## Notebook 4 - Gold Layer Notebook

### Create Widgets

In [ ]:
dbutils.widgets.text("city", "New York")
dbutils.widgets.text("min_revenue", "20")

### Load Silver Data

In [ ]:
from pyspark.sql.functions import sum, avg, count

df = spark.table("retail_job_lab.silver.transactions_enriched")

### Calculate City + Category Revenue

In [ ]:
city_category_df = df.groupBy("city", "category") \
    .agg(sum("total_amount").alias("total_revenue"))

city_category_df.write.mode("overwrite") \
    .saveAsTable("retail_job_lab.gold.city_category_revenue")

### Calculate Store Performance Metrics

In [ ]:
store_perf_df = df.groupBy("store_name") \
    .agg(
        sum("total_amount").alias("total_revenue"),
        avg("total_amount").alias("avg_order_value"),
        count("*").alias("total_transactions")
    )

store_perf_df.write.mode("overwrite") \
    .saveAsTable("retail_job_lab.gold.store_performance")

### Calculate State Level Revenue

In [ ]:
state_revenue_df = df.groupBy("state") \
    .agg(sum("total_amount").alias("total_revenue"))

state_revenue_df.write.mode("overwrite") \
    .saveAsTable("retail_job_lab.gold.state_revenue")

### Read Parameters from Widgets

In [ ]:
city = dbutils.widgets.get("city")
min_revenue = float(dbutils.widgets.get("min_revenue"))

print("City:", city)
print("Min Revenue:", min_revenue)

### Create the Parameterised Table

In [ ]:
from pyspark.sql.functions import col

df = spark.table("retail_job_lab.silver.transactions_enriched")

filtered_df = df.filter(
    (col("city") == city) &
    (col("total_amount") > min_revenue)
)

In [ ]:
parameterised_table_name = f"retail_job_lab.gold.filtered_{city.replace(' ','_')}"

filtered_df.write.mode("overwrite") \
    .saveAsTable(f"retail_job_lab.gold.filtered_{city.replace(' ','_')}")

### Validate Output

In [ ]:
display(spark.table("retail_job_lab.gold.city_category_revenue"))

In [ ]:
display(spark.table("retail_job_lab.gold.store_performance"))

In [ ]:
display(spark.table(f"{parameterised_table_name}"))